# 01. Clean `ami_meter`: physical-plausibility checks + a cleaned table

Before any dataset-description stats, `ami_meter` needs one more pass: some
readings are physically impossible (a metering/pipeline fault), and those
would silently bias any distribution, mean, or compliance check computed on
top of them. This notebook does NOT re-run anything from
`synthetic_ami_creation` (duplicate/inactive circuits, sign correction,
device-model power correction, reconstruction checks) -- all of that
already happened before `ami_meter` was built. This is a second, later
pass, specific to the row-level physical plausibility of the delivered
`V`/`P_kw`/`Q_kvar`/`S_kva`/`power_factor`/`current_a` values themselves.

**Two different kinds of "outlier", handled differently -- see
`lib/ami_clean.py`'s module docstring for the full rationale:**

* **HARD flags** -- physically impossible or internally inconsistent
  (negative current, voltage <= 0V or > 300V, |power factor| > 1, a stored
  `S_kva` that doesn't match V x I for the same row, duplicate/missing key
  fields). These rows are **dropped**.
* **SOFT flags** -- extreme relative to a circuit's own history, but not
  physically impossible (an unusually high per-circuit power reading, which
  could easily be a genuine PV export or EV-charging event). These rows are
  **kept**, just tagged with a boolean column, because CICCADA's own
  research question is AS/NZS 4777.2 Volt-Watt/Volt-VAr compliance and
  curtailment -- an extreme-but-real voltage or power event during a PV
  export spike is exactly the kind of thing this dataset needs to keep for
  later analysis, not delete as noise.

Output: `ami_meter_clean` (cleaned rows, `dt_month`-partitioned Parquet,
same layout as the other Phase 5 tables) written to the local store, plus a
small quality-report CSV and a sample of removed rows saved to
`artefacts/` for manual review.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os

import duckdb
import dotenv
import pandas as pd

from bms_sa_review.ami_analysis.lib import ami_clean as Clean

dotenv.load_dotenv()
STORE_DIR = Path(os.getenv("CICCADA_SYNTHETIC_AMI_DATA_ROOT"))
ARTEFACT_DIR = Path.cwd().parent / "artefacts"
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()


## 1. Load `ami_meter`

Full table, one month at a time, is billions of rows -- see the earlier
discussion in `draft.ipynb` on why loading it whole is a bad idea. This
notebook still needs every row checked, though (a hard-flagged row can't
be found by sampling), so it processes month-by-month rather than loading
everything into memory at once, and accumulates only the small outputs
(quality-report counts, a bounded sample of removed rows) across months.


In [ ]:
MONTHS = con.sql(f"""
    SELECT DISTINCT year, month FROM read_parquet(
        '{(STORE_DIR / "ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    ORDER BY year, month
""").df()
MONTHS_LIST = list(MONTHS.itertuples(index=False, name=None))
print(f"{len(MONTHS_LIST):,} landed (year, month) partitions to process.")
MONTHS


In [ ]:
def read_month(year: int, month: int) -> pd.DataFrame:
    return con.sql(f"""
        SELECT * FROM read_parquet(
            '{(STORE_DIR / "ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
        WHERE year = {year} AND month = {month}
    """).df()


## 2. Run every check, one month at a time

`flags` below is rebuilt fresh each month (a dict of boolean Series aligned
to that month's frame) and fed straight into `apply_cleaning` -- nothing
about the checks themselves depends on seeing more than one month at once,
since every check is either row-local (voltage/PF/current/duplicate/
missing) or scoped to a single circuit's own values (the robust power-
magnitude check), never across months.


In [ ]:
CLEAN_MONTH_DIR = STORE_DIR / "ami_meter_clean"
CLEAN_MONTH_DIR.mkdir(parents=True, exist_ok=True)

quality_reports = []
removed_samples = []
MAX_REMOVED_SAMPLE_ROWS_PER_MONTH = 2000

for year, month in MONTHS_LIST:
    month_frame = read_month(year, month)
    if not len(month_frame):
        continue

    flags = {
        "voltage_implausible": Clean.flag_voltage_outliers(month_frame),
        "power_factor_implausible": Clean.flag_power_factor_outliers(month_frame),
        "current_negative": Clean.flag_negative_current(month_frame),
        "apparent_power_inconsistent": Clean.flag_apparent_power_inconsistency(month_frame),
        "duplicate_reading": Clean.flag_duplicate_readings(month_frame),
        "missing_critical_field": Clean.flag_missing_critical_fields(month_frame),
        "power_magnitude_extreme": Clean.flag_extreme_power_magnitude(month_frame),
    }

    report = Clean.build_quality_report(month_frame, flags)
    report.insert(0, "month", month)
    report.insert(0, "year", year)
    quality_reports.append(report)

    clean_frame, removed_frame = Clean.apply_cleaning(month_frame, flags)

    if len(removed_frame):
        removed_samples.append(removed_frame.sample(
            n=min(MAX_REMOVED_SAMPLE_ROWS_PER_MONTH, len(removed_frame)), random_state=0,
        ))

    part_dir = CLEAN_MONTH_DIR / f"dt_month={year:04d}-{month:02d}"
    part_dir.mkdir(parents=True, exist_ok=True)
    clean_frame.to_parquet(part_dir / "part.parquet", compression="zstd", index=False)

    print(f"{year}-{month:02d}: {len(month_frame):,} rows in, "
          f"{len(removed_frame):,} dropped (hard), {len(clean_frame):,} written.")


## 3. Quality report across every month


In [ ]:
quality_report = pd.concat(quality_reports, ignore_index=True)
quality_report.to_csv(ARTEFACT_DIR / "phase7_ami_meter_quality_report.csv", index=False)

summary = (
    quality_report.groupby(["flag", "kind"], as_index=False)
    .agg(n_flagged=("n_flagged", "sum"))
)
display(summary)


In [ ]:
removed_sample = pd.concat(removed_samples, ignore_index=True) if removed_samples else pd.DataFrame()
removed_sample.to_csv(ARTEFACT_DIR / "phase7_ami_meter_removed_sample.csv", index=False)
print(f"Saved a {len(removed_sample):,}-row sample of dropped (hard-flagged) rows for manual review, "
      f"across {len(removed_samples)} month(s) with at least one drop.")
removed_sample.head(20)


## 4. Sanity-check the cleaned table

Re-open `ami_meter_clean` from disk (not the in-memory frames above) and
confirm row counts and that every HARD flag is now clean, while
`power_magnitude_extreme` (soft) rows are still present as expected.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_clean AS
    SELECT * FROM read_parquet(
        '{CLEAN_MONTH_DIR.as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

before_after = con.sql(f"""
    SELECT
      (SELECT count(*) FROM read_parquet('{(STORE_DIR / "ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)) AS n_rows_before,
      (SELECT count(*) FROM ami_meter_clean) AS n_rows_after,
      (SELECT count(*) FROM ami_meter_clean WHERE power_magnitude_extreme) AS n_soft_flagged_kept
""").df()
before_after


In [ ]:
recheck = con.sql("""
    SELECT
      sum(CASE WHEN V <= 0 OR V > 300 THEN 1 ELSE 0 END) AS n_voltage_implausible,
      sum(CASE WHEN abs(power_factor) > 1.001 THEN 1 ELSE 0 END) AS n_pf_implausible,
      sum(CASE WHEN current_a < 0 THEN 1 ELSE 0 END) AS n_current_negative
    FROM ami_meter_clean
""").df()
assert (recheck.iloc[0] == 0).all(), "a hard-flagged condition survived into ami_meter_clean -- investigate before proceeding"
print("Confirmed: no hard-flagged rows remain in ami_meter_clean.")
recheck
